# Urban LiDAR Analysis

Urban ALS workflows differ from forest workflows in several ways:
- Dense building coverage → PMF often outperforms CSF
- DBSCAN extracts discrete objects (buildings, trees, vehicles)
- RANSAC plane detection segments rooftop geometry
- Point density reveals occlusion (street canyons, bridges)

This notebook covers:
1. PMF ground classification
2. DBSCAN object clustering
3. Rooftop plane detection
4. Point density heatmap

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from occulus.types import AerialCloud, PointCloud

## Synthetic Urban Scene

In [ ]:
rng = np.random.default_rng(13)

# City block: 300x300 m with buildings, trees, roads
n = 80_000
x = rng.uniform(0, 300, n)
y = rng.uniform(0, 300, n)
z = rng.normal(0, 0.05, n)  # flat ground

# 8 buildings of various heights
buildings = [
    (30, 30, 50, 50, 25),   # x1, y1, x2, y2, height
    (60, 30, 100, 80, 40),
    (110, 20, 160, 70, 15),
    (170, 50, 220, 120, 35),
    (30, 120, 90, 180, 20),
    (100, 130, 160, 200, 50),
    (200, 100, 260, 180, 30),
    (50, 220, 150, 280, 45),
]
for bx1, by1, bx2, by2, bh in buildings:
    roof = (x > bx1) & (x < bx2) & (y > by1) & (y < by2)
    z[roof] = bh + rng.normal(0, 0.08, roof.sum())
    # Facade points
    for wx, wy in [(bx1, None), (bx2, None), (None, by1), (None, by2)]:
        if wx is not None:
            wall = (np.abs(x - wx) < 0.8) & (y > by1) & (y < by2)
        else:
            wall = (x > bx1) & (x < bx2) & (np.abs(y - wy) < 0.8)
        z[wall] = rng.uniform(0, bh, wall.sum())

# 20 street trees
for _ in range(20):
    tx, ty, th = rng.uniform(5, 295), rng.uniform(5, 295), rng.uniform(4, 12)
    tree = np.hypot(x - tx, y - ty) < 2.5
    z[tree] += rng.uniform(0, th, tree.sum())

xyz = np.column_stack([x, y, np.maximum(z, 0)]).astype(np.float64)
cloud = AerialCloud(xyz)
print(f'Urban scene: {cloud.n_points:,} points')

## PMF Ground Classification

In [ ]:
from occulus.segmentation import classify_ground_pmf

classified = classify_ground_pmf(cloud)
if classified.classification is not None:
    ground_mask = classified.classification == 2
    above_mask = ~ground_mask
    print(f'Ground        : {ground_mask.sum():,} ({ground_mask.mean():.1%})')
    print(f'Above-ground  : {above_mask.sum():,} ({above_mask.mean():.1%})')

## DBSCAN Object Clustering

In [ ]:
from occulus.segmentation import cluster_dbscan

above_xyz = xyz[above_mask]
above_cloud = PointCloud(above_xyz)

seg = cluster_dbscan(above_cloud, eps=2.0, min_samples=20)
print(f'Objects found : {seg.n_segments}')
print(f'Noise points  : {(seg.labels == -1).sum():,}')

# Size distribution
if seg.n_segments > 0:
    sizes = np.array([(seg.labels == i).sum() for i in range(seg.n_segments)])
    print(f'Avg cluster   : {sizes.mean():.0f} pts')
    print(f'Largest       : {sizes.max():,} pts  (likely large building)')

## Rooftop Plane Detection

In [ ]:
from occulus.features import detect_planes
from occulus.normals import estimate_normals
from occulus.filters import voxel_downsample

above_ds = voxel_downsample(above_cloud, voxel_size=0.5)
above_n = estimate_normals(above_ds, radius=2.0)

planes = detect_planes(above_n, distance_threshold=0.2, min_inliers=50)
print(f'Roof planes detected: {len(planes)}')
for i, p in enumerate(planes[:5]):
    print(f'  Plane {i+1}: {p.n_inliers:,} pts, normal={np.round(p.normal, 3)}')

## Results Dashboard

In [ ]:
from occulus.metrics import point_density

density, xd, yd = point_density(cloud, resolution=5.0)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Ground/above map
axes[0, 0].scatter(xyz[ground_mask, 0], xyz[ground_mask, 1], s=0.3, c='peru', alpha=0.5)
axes[0, 0].scatter(xyz[above_mask, 0], xyz[above_mask, 1], s=0.3, c='royalblue', alpha=0.3)
axes[0, 0].set_title('PMF: Ground (brown) vs Above-ground (blue)')
axes[0, 0].set_xlabel('X (m)'); axes[0, 0].set_ylabel('Y (m)')

# DBSCAN clusters
lbl = seg.labels
noise = lbl == -1
axes[0, 1].scatter(above_xyz[noise, 0], above_xyz[noise, 1], s=0.3, c='lightgray', alpha=0.3)
axes[0, 1].scatter(above_xyz[~noise, 0], above_xyz[~noise, 1],
                   c=lbl[~noise] % 20, cmap='tab20', s=0.5, alpha=0.7)
axes[0, 1].set_title(f'DBSCAN Objects ({seg.n_segments})')
axes[0, 1].set_xlabel('X (m)')

# Density heatmap
im = axes[1, 0].imshow(density.T, origin='lower', extent=[xd[0], xd[-1], yd[0], yd[-1]], cmap='hot')
plt.colorbar(im, ax=axes[1, 0], label='Pts / 25 m²')
axes[1, 0].set_title('Point Density (5 m grid)')
axes[1, 0].set_xlabel('X (m)'); axes[1, 0].set_ylabel('Y (m)')

# Z profile (side view)
axes[1, 1].scatter(xyz[:, 0], xyz[:, 2], s=0.3, c=xyz[:, 2], cmap='viridis', alpha=0.4)
axes[1, 1].set_title('Elevation Profile (X vs Z)')
axes[1, 1].set_xlabel('X (m)'); axes[1, 1].set_ylabel('Z (m)')

plt.suptitle('Urban LiDAR Analysis', fontsize=14)
plt.tight_layout()
plt.savefig('../../outputs/urban_analysis.png', dpi=150)
plt.show()

**Next**: `07_mesh_reconstruction.ipynb`